# 03 · Documents: download procedure documents and extract their text

For each contract with a `documents_link`, this notebook downloads the procedure documents (invitation, specifications,
tender programme, and so on) to `data/docs/<contract_id>/`, extracts their text, and stores it in DuckDB (`documents` table).

**Where the links point** (Covilhã 2026):

| platform | contracts | how its documents are fetched |
|---|---|---|
| `community.vortal.biz` | 827 | public JSON API: link → contract-notice id → document list → one download per file |
| `www.acingov.pt` | 100 | the link itself downloads a ZIP containing every document |
| anogov, espap, compraspt | 11 | not supported yet (logged as `unsupported_platform`) |

**What's inside:** mostly PDFs, plus DOCX, XLSX/XLS, XML and nested ZIPs. This PoC extracts text from **PDF and DOCX**;
other files are saved and listed but not extracted.

**OCR:** in a 513-page sample, only 9 pages had no text layer, and all 9 were technical drawings. So PDFs are
read from their native text first, and only pages with no text are sent to **Tesseract** (Portuguese model, via PyMuPDF).
Set `OCR = False` to skip it. `pages_ocr` counts the pages read by OCR; `pages_without_text` counts the pages still empty after it.

Contracts are processed newest first, `LIMIT` at a time. Every contract is logged in `document_fetch_log`, so
re-running continues where the previous run stopped, and contracts that failed are retried.

**Runs in Google Colab or locally.** The first code cell detects the environment. In Colab it mounts Google Drive and keeps everything under `My Drive/procurement-poc/data/`, so results survive the session.

In [1]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project data
PIP_PACKAGES = ["duckdb>=1.1", "pymupdf>=1.24"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"   # persistent: survives the session
    WORK = Path("/content/work")                                     # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    DATA = Path("data")
    WORK = DATA

DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | data folder: {DATA}")


# DuckDB is slow and can corrupt on a network drive, so in Colab the notebook works on a local copy
# of the database and copies it back to Drive with push_db(). Locally both paths are the same file.
DB_NAME = "procurement.duckdb"
DB_FILE = WORK / DB_NAME


def pull_db():
    """Colab: copy the database from Drive to local disk before connecting."""
    src = DATA / DB_NAME
    if WORK != DATA and src.exists():
        shutil.copy2(src, DB_FILE)
        print(f"database copied from Drive ({src.stat().st_size / 1e6:.1f} MB)")


def push_db(con=None):
    """Colab: save the local copy back to Drive. With an open connection, it checkpoints first."""
    if WORK == DATA:
        return
    if con is not None:
        con.execute("CHECKPOINT")
    tmp = DATA / (DB_NAME + ".tmp")
    shutil.copy2(DB_FILE, tmp)
    tmp.replace(DATA / DB_NAME)
    print(f"database saved to Drive ({DB_FILE.stat().st_size / 1e6:.1f} MB)")


Mounted at /content/drive
running in Colab: True | data folder: /content/drive/MyDrive/procurement-poc/data


In [2]:
import io
import os
import re
import time
import zipfile

import duckdb
import pandas as pd
import pymupdf
import requests

LIMIT = None                 # contracts per run; None = all remaining (about 900 contracts, roughly an hour)
PAUSE_S = 0.5              # pause between requests, to be polite to the platforms
PUSH_EVERY = 25            # Colab: save the database to Drive every N contracts, so a disconnect loses little
DOCS = DATA / "docs"       # downloaded files go straight to Drive in Colab

OCR = True                 # OCR pages that have no text layer (Tesseract)
OCR_LANG = "por"
OCR_DPI = 300
OCR_MAX_PAGES = 60         # per file; a fully scanned 300-page annex would otherwise take ~15 minutes

VORTAL_API = "https://community.vortal.biz/public/api"
http = requests.Session()
http.headers["User-Agent"] = "procurement-poc (UBI MSc research)"
pymupdf.TOOLS.mupdf_display_errors(False)   # some PDFs have minor syntax errors that MuPDF tolerates but reports loudly

pull_db()
con = duckdb.connect(str(DB_FILE))
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    contract_id        BIGINT,
    file_name          VARCHAR,
    file_type          VARCHAR,
    url                VARCHAR,
    local_path         VARCHAR,   -- relative to the data folder, e.g. docs/15645692/Caderno.pdf
    size_bytes         BIGINT,
    pages              INTEGER,
    pages_without_text INTEGER,   -- still empty after OCR
    pages_ocr          INTEGER,   -- pages whose text came from OCR
    status             VARCHAR,   -- extracted | needs_ocr | not_extracted
    text               VARCHAR
);
ALTER TABLE documents ADD COLUMN IF NOT EXISTS pages_ocr INTEGER;   -- databases created before OCR was added
CREATE TABLE IF NOT EXISTS document_fetch_log (
    contract_id BIGINT PRIMARY KEY,
    platform    VARCHAR,
    files       INTEGER,
    status      VARCHAR,          -- ok | no_documents | unsupported_platform | error: ...
    fetched_at  TIMESTAMP
);
""")

database copied from Drive (7.9 MB)


In [3]:
# ── OCR engine ──────────────────────────────────────────────────────────────────────────
# Colab: installs Tesseract with the Portuguese model (about 30 s, once per session).
# Locally on Windows: install Tesseract from https://github.com/UB-Mannheim/tesseract/wiki and tick "Portuguese".
if OCR and IN_COLAB and not shutil.which("tesseract"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-por > /dev/null",
                   shell=True, check=True)


def find_tessdata(lang):
    candidates = [os.environ.get("TESSDATA_PREFIX", ""),
                  *map(str, sorted(Path("/usr/share").glob("tesseract-ocr/*/tessdata"))),
                  "/usr/share/tessdata", r"C:\Program Files\Tesseract-OCR\tessdata"]
    return next((c for c in candidates if c and (Path(c) / f"{lang}.traineddata").exists()), None)


TESSDATA = find_tessdata(OCR_LANG) if OCR else None
if OCR and not TESSDATA:
    print(f"Tesseract with the '{OCR_LANG}' model was not found, so OCR is off for this run")
print("OCR:", f"on ({OCR_LANG}, {TESSDATA})" if TESSDATA else "off")

Tesseract with the 'por' model was not found, so OCR is off for this run
OCR: off


In [4]:
# ── Can this machine reach the platforms? ──────────────────────────────────────────────
# Colab runs on Google servers, usually outside Portugal. If a platform blocks those addresses,
# every contract from it would be logged as an error, so check before the run.
for name, url in [("Vortal API", f"{VORTAL_API}/PublicTenderDocuments/GetPublicTenderInformation"),
                  ("acingov", "https://www.acingov.pt/"),
                  ("dados.gov.pt", "https://dados.gov.pt/api/1/")]:
    try:
        r = http.get(url, timeout=20)
        print(f"{name:13} HTTP {r.status_code}")   # any HTTP answer (even 400/404) means the host is reachable
    except requests.RequestException as e:
        print(f"{name:13} NOT REACHABLE: {type(e).__name__}")

Vortal API    HTTP 500
acingov       HTTP 200
dados.gov.pt  HTTP 200


## 1. Fetchers, one per platform
Each fetcher takes a `documents_link` and returns a list of `(file_name, url, bytes)`.

In [5]:
def fetch_vortal(link):
    uid = link.rstrip("/").split("/")[-1]
    info = http.get(f"{VORTAL_API}/PublicTenderDocuments/GetPublicTenderInformation",
                    params={"uniqueIdentifierEncrypted": uid}, timeout=30).json()
    notice = re.search(r"PT1\.NTC\.\d+", info.get("contractNoticeUrl") or "")
    if not notice:
        return []
    docs = http.get(f"{VORTAL_API}/ContractNoticeDetail/GetContractNoticeDocuments",
                    params={"contractNoticeUId": notice.group(0)}, timeout=30).json() or []
    files = []
    for d in docs:
        time.sleep(PAUSE_S)
        r = http.get(d["downloadUrl"], timeout=120)
        r.raise_for_status()
        files.append((d["name"], d["downloadUrl"], r.content))
    return files


def fetch_acingov(link):
    r = http.get(link, timeout=120)
    r.raise_for_status()
    if not r.content.startswith(b"PK"):      # not a ZIP, e.g. an error page
        return []
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        return [(name, f"{link}#{name}", z.read(name)) for name in z.namelist() if not name.endswith("/")]


FETCHERS = {"community.vortal.biz": fetch_vortal, "www.acingov.pt": fetch_acingov}


def platform_of(link):
    return re.match(r"https?://([^/]+)", link).group(1)

## 2. Text extraction
- **PDF**: the native text layer via PyMuPDF. A page with fewer than `MIN_CHARS_PER_PAGE` characters is OCR'd with Tesseract
  (when `OCR` is on). If every page is still empty, the file is marked `needs_ocr`.
- **DOCX**: the paragraphs in `word/document.xml` (no extra dependency needed).

In [6]:
MIN_CHARS_PER_PAGE = 50   # fewer characters than this = the page is treated as having no text (scan or drawing)


def ocr_page(page):
    textpage = page.get_textpage_ocr(language=OCR_LANG, dpi=OCR_DPI, full=True, tessdata=TESSDATA)
    return page.get_text(textpage=textpage)


def extract_pdf(data):
    pages, empty, ocr_pages = [], 0, 0
    with pymupdf.open(stream=data, filetype="pdf") as doc:
        for page in doc:
            text = page.get_text()
            if len(text.strip()) < MIN_CHARS_PER_PAGE:
                if TESSDATA and ocr_pages < OCR_MAX_PAGES:
                    try:
                        text = ocr_page(page)
                        ocr_pages += 1
                    except Exception:
                        pass
                if len(text.strip()) < MIN_CHARS_PER_PAGE:
                    empty += 1
            pages.append(text)
    status = "needs_ocr" if pages and empty == len(pages) else "extracted"
    return {"text": "\n".join(pages).strip(), "pages": len(pages), "pages_without_text": empty,
            "pages_ocr": ocr_pages, "status": status}


def extract_docx(data):
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        xml = z.read("word/document.xml").decode("utf-8")
    paragraphs = re.split(r"</w:p>", xml)
    text = "\n".join("".join(re.findall(r"<w:t[^>]*>([^<]*)</w:t>", p)) for p in paragraphs)
    return {"text": re.sub(r"\n{3,}", "\n\n", text).strip(), "status": "extracted"}


EXTRACTORS = {"pdf": extract_pdf, "docx": extract_docx}


def extract(file_name, data):
    ext = Path(file_name).suffix.lower().lstrip(".")
    out = {"file_type": ext, "pages": None, "pages_without_text": None, "pages_ocr": None,
           "status": "not_extracted", "text": None}
    if ext in EXTRACTORS:
        try:
            out.update(EXTRACTORS[ext](data))
        except Exception as e:  # a corrupt or password-protected file shouldn't stop the run
            out["status"] = f"not_extracted: {type(e).__name__}"
    return out

## 3. Process one contract
Files are saved to disk first (so they can be re-extracted or OCR'd later without downloading again), then that contract's
rows are replaced in `documents` and it is logged in `document_fetch_log`. `local_path` is stored relative to the data folder,
so the same database works on Windows and in Colab.

In [7]:
def safe_path(contract_id, file_name):
    # platform file names can contain folders ("Processo Concurso/1_Caderno.pdf") and characters Windows rejects
    parts = [re.sub(r'[<>:"\\|?*]', "_", p).strip() for p in file_name.split("/") if p.strip()]
    return Path("docs", str(contract_id), *parts)    # relative to DATA


def process_contract(contract_id, link):
    platform = platform_of(link)
    rows, status = [], "ok"
    if platform not in FETCHERS:
        status = "unsupported_platform"
    else:
        try:
            files = FETCHERS[platform](link)
        except Exception as e:
            files, status = [], f"error: {type(e).__name__}: {e}"[:300]
        if not files and status == "ok":
            status = "no_documents"
        for name, url, data in files:
            rel = safe_path(contract_id, name)
            (DATA / rel).parent.mkdir(parents=True, exist_ok=True)
            (DATA / rel).write_bytes(data)
            rows.append({"contract_id": contract_id, "file_name": name, "url": url,
                         "local_path": rel.as_posix(), "size_bytes": len(data), **extract(name, data)})

    con.execute("BEGIN TRANSACTION")
    con.execute("DELETE FROM documents WHERE contract_id = ?", [contract_id])
    if rows:
        batch = pd.DataFrame(rows)
        con.execute("INSERT INTO documents BY NAME SELECT * FROM batch")
    con.execute("INSERT OR REPLACE INTO document_fetch_log VALUES (?, ?, ?, ?, ?)",
                [contract_id, platform, len(rows), status, pd.Timestamp.now()])
    con.execute("COMMIT")
    return status, len(rows)

## 4. Run
Picks the contracts not yet fetched successfully, newest first.

In [8]:
todo = con.sql("""
    SELECT contract_id, documents_link
    FROM contracts
    WHERE documents_link IS NOT NULL
      AND contract_id NOT IN (SELECT contract_id FROM document_fetch_log WHERE status NOT LIKE 'error%')
    ORDER BY publication_date DESC
""").df()
batch_todo = todo.head(LIMIT) if LIMIT else todo
print(f"{len(todo)} contracts still to fetch, processing {len(batch_todo)} now")

start = time.time()
for i, (cid, link) in enumerate(batch_todo.itertuples(index=False), 1):
    status, n = process_contract(cid, link)
    print(f"[{i}/{len(batch_todo)}] {cid} {platform_of(link):22} {n:3} files  {status}")
    time.sleep(PAUSE_S)
    if i % PUSH_EVERY == 0 and i < len(batch_todo):
        push_db(con)
print(f"done in {time.time() - start:.0f}s")
push_db(con)

971 contracts still to fetch, processing 971 now
[1/971] 15730625 community.vortal.biz     0 files  no_documents
[2/971] 15730686 community.vortal.biz     7 files  ok
[3/971] 15730703 community.vortal.biz     0 files  no_documents
[4/971] 15730712 community.vortal.biz     0 files  no_documents
[5/971] 15730751 community.vortal.biz     0 files  no_documents
[6/971] 15730783 community.vortal.biz     0 files  no_documents
[7/971] 15730813 community.vortal.biz     0 files  no_documents
[8/971] 15730853 community.vortal.biz     0 files  no_documents
[9/971] 15730868 community.vortal.biz     0 files  no_documents
[10/971] 15731236 www.anogov.com           0 files  unsupported_platform
[11/971] 15731675 www.anogov.com           0 files  unsupported_platform
[12/971] 15724789 community.vortal.biz     0 files  error: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
[13/971] 15724961 community.vortal.biz     0 files  error: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
[14

## 5. What was collected

In [9]:
con.sql("""
    SELECT platform, status, COUNT(*) AS contracts, SUM(files) AS files
    FROM document_fetch_log GROUP BY ALL ORDER BY contracts DESC
""").df()

,platform,status,contracts,files
0,community.vortal.biz,error: JSONDecodeError: Expecting value: line ...,600,0.0
1,community.vortal.biz,ok,190,833.0
2,www.acingov.pt,ok,101,279.0
3,community.vortal.biz,no_documents,83,0.0
4,www.anogov.com,unsupported_platform,9,0.0
5,www.compraspt.com,unsupported_platform,2,0.0
6,plataforma-sncp.espap.gov.pt,unsupported_platform,2,0.0
7,www.acingov.pt,no_documents,1,0.0


In [10]:
con.sql("""
    SELECT file_type, status, COUNT(*) AS files, SUM(pages) AS pages,
           SUM(pages_ocr) AS pages_ocr, SUM(pages_without_text) AS pages_without_text,
           ROUND(SUM(size_bytes) / 1e6, 1) AS mb, SUM(length(text)) AS chars
    FROM documents GROUP BY ALL ORDER BY files DESC
""").df()

,file_type,status,files,pages,pages_ocr,pages_without_text,mb,chars
0,pdf,extracted,829,11004.0,0.0,95.0,568.3,21899387.0
1,docx,extracted,91,NaN,NaN,NaN,3.7,358784.0
2,xlsx,not_extracted,75,NaN,NaN,NaN,6.4,NaN
3,pdf,needs_ocr,59,739.0,0.0,739.0,295.5,115.0
4,rar,not_extracted,31,NaN,NaN,NaN,3313.9,NaN
5,zip,not_extracted,15,NaN,NaN,NaN,8.1,NaN
6,xls,not_extracted,8,NaN,NaN,NaN,0.3,NaN
7,dwf,not_extracted,2,NaN,NaN,NaN,3.3,NaN
8,,not_extracted,1,NaN,NaN,NaN,0.5,NaN
9,xml,not_extracted,1,NaN,NaN,NaN,0.3,NaN


In [11]:
# Example: pull the start of one contract's specifications (caderno de encargos)
row = con.sql("""
    SELECT d.contract_id, c.object, d.file_name, d.text
    FROM documents d JOIN contracts c USING (contract_id)
    WHERE d.status = 'extracted' AND d.file_name ILIKE '%encargos%'
    LIMIT 1
""").fetchall()
if row:
    row = row[0]
    print(row[0], "|", row[1]); print(row[2]); print("-" * 80); print(row[3][:1500])

12446543 | EMPREITADA DE OBRAS DE REABILITAÇÃO DE PAVIMENTOS BETUMINOSOS NAS RUAS ALDEIA DA NEVE, URZE E NA TRAVESSA DA URZE - PENHAS DA SAÚDE
03 CadernoEncargos_C.Técnicas.pdf
--------------------------------------------------------------------------------
JULHO 2025 
 
Designação da Empreitada 
“Reabilitação de Pavimentos Betuminosos nas 
Ruas Aldeia da Neve, Urze e na travessa da Urze - 
Penhas da Saúde” 
 
Caderno de Encargos  
– 
 Cláusulas Técnicas Gerais  
e 
Cláusulas Técnicas Especiais 
 

                     
 
 
 
 
“Empreitada de Reabilitação de Pavimentos Betuminosos nas Ruas, Aldeia 
da Neve, Urze e Travessa da Urze - Penhas da Saúde” 
 
CONDIÇÕES TÉCNICAS GERAIS E CONDIÇÕES TÉCNICAS ESPECIAIS                                         Página 2 de 41 
 
 
 
 
ÍNDICE 
CONDIÇÕES TÉCNICAS GERAIS ................................................................................................................................ 4 
1. 
INTRODUÇÃO .....................................

## 6. Re-extract files already on disk (optional)
Runs the extraction again on PDFs that still have pages without text, reading them from `data/docs/` instead of
downloading them again. Use it after turning OCR on, or after changing `MIN_CHARS_PER_PAGE`.

In [12]:
def resolve(local_path):
    # also accepts paths written by the earlier Windows runs, e.g. "data\\docs\\123\\file.pdf"
    p = local_path.replace("\\", "/")
    return DATA / (p[len("data/"):] if p.startswith("data/") else p)


pending = con.sql("""
    SELECT contract_id, file_name, local_path FROM documents
    WHERE file_type = 'pdf' AND (pages_without_text > 0 OR status = 'needs_ocr')
""").fetchall()
print(f"{len(pending)} PDFs with pages still lacking text")

if TESSDATA:
    for cid, name, local_path in pending:
        path = resolve(local_path)
        if not path.exists():
            print("missing on disk:", path)
            continue
        r = extract(name, path.read_bytes())
        con.execute("""UPDATE documents SET text = ?, pages = ?, pages_without_text = ?, pages_ocr = ?, status = ?
                       WHERE contract_id = ? AND local_path = ?""",
                    [r["text"], r["pages"], r["pages_without_text"], r["pages_ocr"], r["status"], cid, local_path])
        print(f"{cid} {name[:60]:60} OCR pages: {r['pages_ocr']}, still empty: {r['pages_without_text']}")
    push_db(con)

80 PDFs with pages still lacking text


In [13]:
con.close()
push_db()   # Colab: copy the database back to Drive

database saved to Drive (53.2 MB)
